In [ ]:
# Ch08: 서비스화·시각화·윤리
import sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                          'pandapower', 'gradio', 'networkx', 'seaborn', 'plotly'])
import pandapower as pp
import pandapower.networks as pn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline

np.random.seed(42)

try:
    sys.path.insert(0, '..')
    from utils.style import set_style, COLORS
    set_style()
except:
    COLORS = {'navy': '#2C3E50', 'teal': '#1B7A8A', 'amber': '#D4984A', 'sage': '#5A7D6A', 'coral': '#C75C3A'}

## 1. Streamlit → Gradio 변환: OPF 대시보드

교재의 Streamlit 코드를 Gradio로 변환하여 Colab에서 인라인 실행합니다.

In [ ]:
import gradio as gr

# 원래 Streamlit 코드 (참조용):
# st.title("IEEE 14-Bus OPF 대시보드")
# load_scale = st.sidebar.slider("부하 배율", 0.5, 2.0, 1.0, 0.1)
# net = pn.case14(); net.load["p_mw"] *= load_scale
# pp.runopp(net)

def run_opf_dashboard(load_scale):
    """OPF 시뮬레이션 및 시각화"""
    net = pn.case14()
    net.load["p_mw"] *= load_scale
    net.load["q_mvar"] *= load_scale
    
    try:
        pp.runopp(net, verbose=False)
    except:
        pp.runpp(net, verbose=False)
    
    # 시각화
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # 전압 프로파일
    voltages = net.res_bus.vm_pu.values
    colors = ['#C75C3A' if v < 0.95 or v > 1.05 else '#1B7A8A' for v in voltages]
    axes[0].bar(net.res_bus.index, voltages, color=colors)
    axes[0].axhline(y=0.95, color='#D4984A', ls='--')
    axes[0].axhline(y=1.05, color='#D4984A', ls='--')
    axes[0].set_xlabel('모선 번호')
    axes[0].set_ylabel('전압 (p.u.)')
    axes[0].set_title('모선별 전압')
    
    # 선로 부하율
    loading = net.res_line.loading_percent.values
    axes[1].barh(net.res_line.index, loading, color='#5A7D6A')
    axes[1].axvline(x=100, color='#C75C3A', ls='--')
    axes[1].set_xlabel('부하율 (%)')
    axes[1].set_ylabel('선로 번호')
    axes[1].set_title('선로 부하율')
    
    plt.tight_layout()
    
    cost = getattr(net, 'res_cost', 0)
    summary = f"총 비용: {cost:.0f} $/h | 최저 전압: {voltages.min():.4f} p.u. | 최대 부하율: {loading.max():.1f}%"
    
    return fig, summary

demo = gr.Interface(
    fn=run_opf_dashboard,
    inputs=gr.Slider(0.5, 2.0, value=1.0, step=0.1, label="부하 배율"),
    outputs=[gr.Plot(label="시각화"), gr.Textbox(label="결과 요약")],
    title="IEEE 14-Bus OPF 대시보드 (Gradio)",
    description="부하 배율을 조정하여 OPF 결과를 확인합니다."
)

demo.launch(inline=True, share=False)

## 2. FastAPI → In-notebook 데모

In [ ]:
# FastAPI 서버를 노트북에서 시뮬레이션
# (실제 서버 대신 함수 직접 호출 방식)

from dataclasses import dataclass, field

@dataclass
class LoadChange:
    bus_id: int
    p_mw: float
    q_mvar: float = 0.0

def power_flow_api(changes: list) -> dict:
    """FastAPI /power-flow 엔드포인트 시뮬레이션"""
    net = pn.case14()
    for c in changes:
        mask = net.load["bus"] == c.bus_id
        if mask.any():
            net.load.loc[mask, "p_mw"] += c.p_mw
            net.load.loc[mask, "q_mvar"] += c.q_mvar
    pp.runpp(net)
    return {
        "bus_voltages": {int(i): round(float(v), 5) for i, v in net.res_bus.vm_pu.items()},
        "line_loadings": {int(i): round(float(l), 2) for i, l in net.res_line.loading_percent.items()},
        "converged": bool(net.converged),
    }

# 테스트 호출
result = power_flow_api([LoadChange(bus_id=4, p_mw=5.0, q_mvar=1.0)])
print("=== Power Flow API 응답 ===")
print(f"수렴: {result['converged']}")
print(f"전압 범위: {min(result['bus_voltages'].values()):.4f} ~ {max(result['bus_voltages'].values()):.4f} p.u.")
print(f"\n참고: 실제 배포 시 FastAPI + uvicorn 사용")
print("  uvicorn server:app --reload")
print("  http://localhost:8000/docs (Swagger UI)")

## 3. 논문 품질 matplotlib 스타일

In [ ]:
# TODO: IEEE 논문 스타일 설정을 작성하세요
# 힌트: plt.rcParams.update({...})
# 필수 항목: figure.figsize, axes.spines.top/right, savefig.dpi, grid

# plt.rcParams.update({...})  # <-- 설정을 작성하세요
pass

# 색상 팔레트 정의
# COLORS = {...}  # <-- 5가지 색상을 정의하세요

## 4. 계통도 시각화 (NetworkX)

In [ ]:
import networkx as nx

net = pn.case14()
pp.runpp(net)

G = nx.Graph()
for i in net.bus.index:
    G.add_node(i)
for _, row in net.line.iterrows():
    G.add_edge(int(row['from_bus']), int(row['to_bus']),
               loading=net.res_line.at[row.name, 'loading_percent'])

pos = nx.spring_layout(G, seed=42)
v_dev = [abs(net.res_bus.at[n, 'vm_pu'] - 1.0) * 2000 + 100 for n in G.nodes()]

edge_colors = []
for u, v, d in G.edges(data=True):
    if d['loading'] > 60:
        edge_colors.append('#C75C3A')
    elif d['loading'] > 40:
        edge_colors.append('#D4984A')
    else:
        edge_colors.append('#5A7D6A')

fig, ax = plt.subplots(figsize=(10, 8))
nx.draw(G, pos, ax=ax, node_size=v_dev, edge_color=edge_colors,
        width=2, with_labels=True, font_size=8, font_weight='bold',
        node_color='#1B7A8A', font_color='white')
ax.set_title('IEEE 14-bus 계통도\n(노드 크기 = 전압 편차, 선 색상 = 부하율)')
plt.tight_layout()
plt.show()

## 5. 조류 흐름 히트맵

In [ ]:
import seaborn as sns

flow_data = pd.DataFrame({
    'from→to': [f"{int(r['from_bus'])}→{int(r['to_bus'])}" for _, r in net.line.iterrows()],
    'P (MW)': net.res_line['p_from_mw'].values,
    'Q (Mvar)': net.res_line['q_from_mvar'].values,
})
flow_data = flow_data.set_index('from→to')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(flow_data[['P (MW)']].T, ax=axes[0], cmap='RdBu_r',
            annot=True, fmt='.1f', linewidths=0.5)
axes[0].set_title('유효전력 조류 (MW)')

sns.heatmap(flow_data[['Q (Mvar)']].T, ax=axes[1], cmap='PiYG_r',
            annot=True, fmt='.1f', linewidths=0.5)
axes[1].set_title('무효전력 조류 (Mvar)')

plt.tight_layout()
plt.show()

## 6. Plotly 인터랙티브 시계열 (Fig 8.6)

1주일 전력 수급 시계열: 총 수요, 태양광, 풍력, 순수요를 Plotly로 인터랙티브하게 시각화합니다.

In [ ]:
import plotly.graph_objects as go

hours = np.arange(168)  # 1주일
demand = 300 + 80*np.sin(hours*2*np.pi/24) + 20*np.random.randn(168)
solar = np.tile(np.maximum(0, 60*np.sin((np.arange(24)-6)*np.pi/12)), 7)
wind = 40 + 20*np.sin(hours*2*np.pi/72) + 10*np.random.randn(168)

fig = go.Figure()
fig.add_trace(go.Scatter(y=demand, fill='tozeroy', name='총 수요'))
fig.add_trace(go.Scatter(y=solar, name='태양광'))
fig.add_trace(go.Scatter(y=np.maximum(0, wind), name='풍력'))
fig.add_trace(go.Scatter(y=demand-solar-wind, name='순수요'))

fig.update_layout(
    title='Fig 8.6: 1주일 전력 수급 시계열',
    xaxis_title='시간 (h)',
    yaxis_title='전력 (MW)',
    hovermode='x unified',
    template='plotly_white',
)
fig.show()